Do laboratório à equação diferencial: uma sensibilização sobre significados

In [ ]:
"""
Sistema reativo mínimo para notebooks interativos
==================================================

Este módulo implementa um pequeno grafo reativo baseado em **nós** (*nodes*)
e **listeners**.  O objetivo é permitir que widgets do :mod:`ipywidgets`
reajam automaticamente a mudanças de valor, sem acoplamento direto entre
produtores e consumidores.

Conceito central — listeners
-----------------------------
Um *listener* é uma função com a assinatura ``(value: Any) -> None`` que é
*registrada* em um :class:`ReactiveNode` via :meth:`ReactiveNode.subscribe`.
Sempre que :meth:`ReactiveNode.set` é chamado, o nó itera sobre todos os
listeners cadastrados e os invoca com o novo valor.

A maneira padrão de tornar um widget reativo é envolver a atualização
desejada do widget em uma função e registrá-la como listener::

    node = ReactiveNode(0)
    label = widgets.Label()

    # Registra um listener que mantém o texto do label sincronizado
    node.subscribe(lambda v: setattr(label, "value", str(v)))

    node.set(42)   # dispara o listener → label exibe "42"

O método de conveniência :meth:`ReactiveNode.to_widget` encapsula exatamente
esse padrão, eliminando o :func:`setattr` explícito::

    node.to_widget(label, "value", transform=str)

Para criar nós derivados que reagem a outros nós (compondo o grafo),
utilize :meth:`ReactiveNode.map` (um nó de entrada) ou :func:`computed`
(múltiplos nós de entrada).

Fluxo resumido
--------------
::

    WidgetNode  ──set()──►  ReactiveNode  ──set()──►  ReactiveNode
    (fonte)                 (derivado via map/computed) (saída → widget)
                                 │
                            listeners notificados
                            em cadeia

Classes e funções principais
-----------------------------
- :class:`ReactiveNode` — nó base com suporte a listeners.
- :class:`WidgetNode`   — nó reativo cuja *fonte* é um widget observável.
- :func:`bind`          — atalho para criar um :class:`WidgetNode`.
- :func:`computed`      — nó derivado de múltiplas dependências.
"""
from __future__ import annotations

from collections.abc import Callable
from dataclasses import dataclass, field
from typing import Any

import ipywidgets as widgets


type Listener = Callable[[Any], None]
"""Tipo de um listener: função que recebe o novo valor de um nó."""

type Transform = Callable[..., Any]
"""Tipo de uma transformação aplicada a valores de nós."""


def identity(x: Any) -> Any:
    """Retorna o argumento sem modificações.

    Usada como transformação padrão em :meth:`ReactiveNode.to_widget`.

    Parameters
    ----------
    x : Any
        Valor a ser retornado.

    Returns
    -------
    Any
        O mesmo objeto recebido.
    """
    return x


@dataclass(slots=True)
class ReactiveNode:
    """
    Nó reativo que armazena um valor e notifica listeners quando esse valor muda.

    Esta classe é o bloco fundamental do sistema reativo.  Um nó pode
    desempenhar três papéis:

    - **fonte primária** — alimentado externamente (e.g. por um :class:`WidgetNode`);
    - **valor derivado** — calculado a partir de outros nós via :meth:`map` ou
      :func:`computed`;
    - **produtor de saída** — direciona seu valor a um widget via
      :meth:`to_widget`.

    Mecanismo de listeners
    ----------------------
    Um *listener* é qualquer callable ``(value: Any) -> None`` registrado com
    :meth:`subscribe`.  A lista de listeners é mantida em :attr:`listeners`.

    Toda vez que :meth:`set` é chamado:

    1. O atributo :attr:`value` é atualizado.
    2. :meth:`_notify` percorre :attr:`listeners` e invoca cada um com o novo
       valor.

    Isso cria uma cadeia de propagação: alterar um nó raiz dispara, em cascata,
    todos os nós e widgets que dependem dele.

    Para tornar um widget reativo, basta criar uma função que o atualize e
    registrá-la como listener::

        node = ReactiveNode(0)
        slider_out = widgets.IntSlider()

        node.subscribe(lambda v: setattr(slider_out, "value", int(v)))

        node.set(7)   # slider_out.value passa a ser 7

    O método :meth:`to_widget` encapsula esse padrão de forma declarativa.

    Parameters
    ----------
    value : Any, optional
        Valor inicial do nó.  Padrão: ``None``.
    listeners : list[Listener], optional
        Lista de callbacks pré-registrados.  Em geral não é fornecida
        diretamente; use :meth:`subscribe` para registrar listeners após a
        criação.

    Examples
    --------
    Nó simples com listener impresso no console:

    >>> node = ReactiveNode(0)
    >>> node.subscribe(lambda v: print(f"novo valor: {v}"))
    ReactiveNode(value=0, listeners=[...])
    >>> node.set(3)
    novo valor: 3

    Ver também
    ----------
    WidgetNode : nó reativo cuja fonte é um widget do ipywidgets.
    computed   : cria um nó derivado de múltiplas dependências.
    """

    value: Any = None
    listeners: list[Listener] = field(default_factory=list)

    def get(self) -> Any:
        """
        Devolve o valor atual do nó.

        Returns
        -------
        Any
            Valor atualmente armazenado em :attr:`value`.
        """
        return self.value

    def set(self, value: Any) -> None:
        """
        Atualiza o valor do nó e notifica todos os listeners registrados.

        A notificação é síncrona e segue a ordem de registro dos listeners.
        Listeners adicionados durante a notificação *não* serão chamados na
        rodada atual (a lista é copiada antes da iteração).

        Parameters
        ----------
        value : Any
            Novo valor do nó.

        See Also
        --------
        subscribe : registra um listener para ser chamado por este método.
        _notify   : implementação interna da notificação.
        """
        self.value = value
        self._notify()

    def subscribe(
        self,
        listener: Listener,
        *,
        call_immediately: bool = False,
    ) -> ReactiveNode:
        """
        Registra um listener para reagir às mudanças deste nó.

        O listener é adicionado ao final de :attr:`listeners` e será chamado
        em cada invocação de :meth:`set`.

        Para tornar um widget reativo, passe uma função que atualize o widget::

            progress = widgets.FloatProgress()
            node.subscribe(lambda v: setattr(progress, "value", v))

        Parameters
        ----------
        listener : Listener
            Callable ``(value: Any) -> None`` a ser registrado.
        call_immediately : bool, optional
            Se ``True``, o listener é chamado imediatamente com o valor atual
            logo após ser registrado.  Útil para inicializar a interface sem
            precisar forçar um :meth:`set` manual.  Padrão: ``False``.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento de chamadas::

                node.subscribe(fn_a).subscribe(fn_b)

        See Also
        --------
        unsubscribe : remove um listener previamente registrado.
        to_widget   : atalho que cria e registra um listener para um widget.
        """
        self.listeners.append(listener)
        if call_immediately:
            listener(self.value)
        return self

    def unsubscribe(self, listener: Listener) -> None:
        """
        Remove um listener previamente registrado.

        Se o listener não estiver na lista, a chamada é ignorada silenciosamente.

        Parameters
        ----------
        listener : Listener
            O mesmo callable passado a :meth:`subscribe`.
        """
        if listener in self.listeners:
            self.listeners.remove(listener)

    def _notify(self) -> None:
        """
        Invoca todos os listeners com o valor atual do nó.

        A lista de listeners é copiada antes da iteração para permitir que um
        listener se cancele (via :meth:`unsubscribe`) sem causar erros de
        modificação concorrente.

        Notes
        -----
        Método de uso interno; prefira :meth:`set` para acionar a propagação.
        """
        for listener in list(self.listeners):
            listener(self.value)

    def map(
        self,
        func: Callable[[Any], Any],
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Cria um nó filho derivado deste, transformando seu valor.

        Internamente, registra um listener em ``self`` que chama :meth:`set`
        no nó filho sempre que o valor pai mudar.

        Parameters
        ----------
        func : Callable[[Any], Any]
            Função de transformação unária aplicada ao valor deste nó.
        initialize : bool, optional
            Se ``True``, o nó filho é inicializado imediatamente com
            ``func(self.value)``.  Padrão: ``True``.

        Returns
        -------
        ReactiveNode
            Novo nó derivado.  Qualquer listener registrado nele será
            notificado quando *este* nó mudar.

        Examples
        --------
        >>> x = ReactiveNode(3)
        >>> x_sq = x.map(lambda v: v ** 2)
        >>> x_sq.get()
        9
        >>> x.set(5)
        >>> x_sq.get()
        25

        See Also
        --------
        computed : versão que aceita múltiplos nós de entrada.
        """
        initial = func(self.value) if initialize else None
        child = ReactiveNode(initial)

        def listener(value: Any) -> None:
            child.set(func(value))

        self.subscribe(listener)
        return child

    def to_widget(
        self,
        widget: widgets.Widget,
        attr: str,
        transform: Callable[[Any], Any] = identity,
        *,
        initialize: bool = True,
    ) -> ReactiveNode:
        """
        Liga este nó a um atributo de um widget, tornando-o reativo.

        Este método encapsula o padrão manual de registrar um listener que
        atualiza um widget::

            # Equivalente explícito ao que to_widget faz internamente:
            node.subscribe(lambda v: setattr(widget, attr, transform(v)),
                           call_immediately=initialize)

        Parameters
        ----------
        widget : widgets.Widget
            Widget de destino a ser atualizado.
        attr : str
            Nome do atributo do widget a ser escrito, por exemplo ``"value"``,
            ``"description"`` ou ``"style"``.
        transform : Callable[[Any], Any], optional
            Transformação aplicada ao valor do nó antes da atribuição.
            Padrão: :func:`identity` (nenhuma transformação).
        initialize : bool, optional
            Se ``True``, o atributo do widget é definido imediatamente com o
            valor atual do nó (sem necessidade de um :meth:`set` inicial).
            Padrão: ``True``.

        Returns
        -------
        ReactiveNode
            O próprio nó, permitindo encadeamento::

                node.to_widget(w1, "value").to_widget(w2, "description", str)

        Examples
        --------
        Exibir o quadrado do valor de um slider em uma barra de progresso::

            x_sq.to_widget(progress, "value")

        Exibir o valor como texto formatado em um label::

            x.to_widget(label, "value", transform=lambda v: f"x = {v:.2f}")

        See Also
        --------
        subscribe : método de baixo nível para registrar listeners manualmente.
        """
        def listener(value: Any) -> None:
            setattr(widget, attr, transform(value))

        self.subscribe(listener, call_immediately=initialize)
        return self


class WidgetNode(ReactiveNode):
    """
    Nó reativo cuja *fonte* é um atributo observável de um widget ipywidgets.

    :class:`WidgetNode` é a ponte de entrada do sistema reativo: ele observa
    um widget (tipicamente um controle interativo como um slider) e propaga
    as mudanças do usuário para os listeners registrados, que por sua vez podem
    atualizar outros widgets ou recalcular valores derivados.

    Internamente, o nó usa :meth:`widgets.Widget.observe` para escutar
    alterações no atributo indicado e chama :meth:`ReactiveNode.set` sempre
    que o widget muda — o que aciona toda a cadeia de listeners cadastrados.

    Parameters
    ----------
    widget : widgets.Widget
        Widget de origem cujos valores serão observados.
    attr : str
        Nome do atributo observado, normalmente ``"value"``.

    Examples
    --------
    Criar um nó a partir de um slider e registrar um listener::

        slider = widgets.FloatSlider(min=0, max=10, value=5)
        node = WidgetNode(slider, "value")
        node.subscribe(lambda v: print(f"slider movido para {v}"))

    Na prática, use a função :func:`bind` como atalho::

        node = bind(slider)

    See Also
    --------
    bind : função fábrica de conveniência para criar um :class:`WidgetNode`.
    """

    def __init__(self, widget: widgets.Widget, attr: str) -> None:
        self.widget = widget
        self.attr = attr
        self._widget_handler = self._make_handler()
        super().__init__(getattr(widget, attr))
        widget.observe(self._widget_handler, names=attr)

    def _make_handler(self) -> Callable[[dict[str, Any]], None]:
        """
        Cria o callback interno que traduz eventos do ipywidgets para :meth:`set`.

        O ipywidgets entrega um dicionário ``change`` com chaves como ``"new"``
        e ``"old"``; este handler extrai ``change["new"]`` e o passa a
        :meth:`ReactiveNode.set`, disparando todos os listeners do nó.

        Returns
        -------
        Callable[[dict[str, Any]], None]
            Função compatível com a assinatura esperada por
            :meth:`widgets.Widget.observe`.
        """
        def handler(change: dict[str, Any]) -> None:
            self.set(change["new"])

        return handler

    def unlink(self) -> None:
        """
        Remove a observação do widget de origem, desativando o nó.

        Após chamar este método, mudanças no widget não propagarão mais
        atualizações pelos listeners.  Útil para evitar vazamentos de memória
        ao descartar um painel interativo.
        """
        self.widget.unobserve(self._widget_handler, names=self.attr)


def bind(widget: widgets.Widget, attr: str = "value") -> WidgetNode:
    """
    Cria um :class:`WidgetNode` a partir de um atributo de um widget.

    Função fábrica de conveniência que evita a instanciação explícita de
    :class:`WidgetNode`.

    Parameters
    ----------
    widget : widgets.Widget
        Widget de origem a ser observado.
    attr : str, optional
        Atributo observado.  Padrão: ``"value"``, que corresponde ao valor
        principal da maioria dos widgets interativos (sliders, caixas de texto,
        etc.).

    Returns
    -------
    WidgetNode
        Nó reativo vinculado ao widget.

    Examples
    --------
    ::

        slider = widgets.FloatSlider(min=0, max=100, value=50)
        x = bind(slider)
        x.subscribe(lambda v: print(v))   # imprime cada vez que o slider muda

    See Also
    --------
    WidgetNode : classe subjacente criada por esta função.
    computed   : combina múltiplos nós em um único nó derivado.
    """
    return WidgetNode(widget, attr)


def computed(
    *nodes: ReactiveNode,
    func: Transform,
    initialize: bool = True,
) -> ReactiveNode:
    """
    Cria um nó derivado de um ou mais nós de entrada.

    Para cada nó de entrada é registrado um listener que recalcula o valor
    derivado chamando ``func`` com os valores atuais de *todas* as
    dependências.  Assim, qualquer mudança em qualquer dependência propaga-se
    automaticamente para o nó resultante e, por conseguinte, para os listeners
    registrados nele.

    Parameters
    ----------
    *nodes : ReactiveNode
        Um ou mais nós de entrada (dependências).
    func : Transform
        Função que combina os valores das dependências na ordem em que foram
        passados::

            resultado = func(nodes[0].get(), nodes[1].get(), ...)

    initialize : bool, optional
        Se ``True``, o nó derivado é inicializado imediatamente avaliando
        ``func`` com os valores atuais das dependências.  Padrão: ``True``.

    Returns
    -------
    ReactiveNode
        Novo nó cujo valor é sempre ``func(*[n.get() for n in nodes])``,
        atualizado toda vez que qualquer dependência mudar.

    Examples
    --------
    Nó que representa a soma de dois sliders::

        a = bind(widgets.IntSlider(value=1))
        b = bind(widgets.IntSlider(value=2))
        soma = computed(a, b, func=lambda x, y: x + y)
        soma.to_widget(label, "value", transform=str)

    See Also
    --------
    ReactiveNode.map : versão simplificada para um único nó de entrada.
    bind             : cria um nó de entrada a partir de um widget.
    """
    def evaluate() -> Any:
        return func(*(node.get() for node in nodes))

    result = ReactiveNode(evaluate() if initialize else None)

    def recompute(_: Any) -> None:
        result.set(evaluate())

    for node in nodes:
        node.subscribe(recompute)

    return result


In [11]:
import ipywidgets as widgets
from IPython.display import display

xslider = widgets.FloatSlider(min=0, max=100, step=0.1, value=50, description="x")
yprogress = widgets.FloatProgress(min=0, max=10000, description="x²")

x = bind(xslider, "value")
x_squared = x.map(lambda v: v**2)

x_squared.to_widget(yprogress, "value")

display(xslider)
display(yprogress)

FloatSlider(value=50.0, description='x')

FloatProgress(value=2500.0, description='x²', max=10000.0)